# CZNR on IonQ Hardware

*CliNR in Practice: Reaching Breakeven with CZNR* ran CZNR with the corrections applied inside the circuit, using mid-circuit measurement and feedforward. Real trapped-ion hardware of the kind we target here does not offer mid-circuit measurement, so this notebook uses the equivalent approach that the hardware experiment of Tham and Delfosse (arXiv:2504.13356) actually uses: run the circuit to the end, measure every qubit once, and apply the restarts and Pauli corrections afterwards in classical post-processing.

The previous notebook already showed, in simulation, that this post-processing version reaches the same logical error rate as the in-circuit protocol. Here we take that as given and focus on getting the circuit onto hardware: we build it once, draw it, set up the post-processing correction, and prepare the IonQ Forte submission through qBraid. The actual hardware call is gated behind a flag, so the notebook is safe to run end to end without spending credits.

### What this notebook does

1. Builds the CZNR circuit once, parametrized by the number of qubits per register, and draws it at three qubits so the structure is visible.
2. Sets up the post-processing correction that replaces the in-circuit feedforward of the previous notebook.
3. Prepares the qBraid to IonQ Forte submission, gated behind a flag, with the steps that need a live account marked clearly.
4. Performs the progressive stabilizer-ignoring analysis on hardware results, when a hardware run is available.

For the simulated breakeven evidence that post-processing matches the in-circuit protocol, see the previous notebook. The circuit and its diagram use Qiskit; the post-processing correction uses [Stim](https://github.com/quantumlib/Stim)'s Clifford tableaus.

## Setup

In [ ]:
%pip install stim numpy matplotlib qiskit pylatexenc qbraid --quiet

import stim
import numpy as np
import matplotlib.pyplot as plt

## The CZNR circuit

The experiment runs the Clifford `C` *squared*, which uses two resource graph states. The data starts on the input register and is passed through the two resources in turn:

1. Prepare the input `Cprep|0>` on the input register, and a graph state `C|+>` on each of the two resource registers.
2. Verify both resource states by measuring stabilizers, read out on separate ancilla qubits that are left idle until the end — there is no mid-circuit measurement.
3. Apply `C` once by coupling the first resource into the input, then apply `C` again by coupling the second resource into the first. The output ends up on the second resource register.
4. Undo the input preparation on the second resource register.
5. Measure every qubit in the Z basis at the end (on IonQ this happens automatically).

We write this as a single function, `build_cznr(n, ...)`, parametrized by the number of qubits per register. It is drawn below at `n = 3` with one stabilizer per resource so the wiring is legible, and the *same* function is submitted at `n = 10` — the version used in the experiment, with the same structure and more CZ gates. The protocol itself, and its origin in the graph-state formulation of CliNR, is developed in the previous notebook, *CliNR in Practice: Reaching Breakeven with CZNR*.

## Building blocks

The Clifford is the all-pairs CZ circuit on `N` qubits, run as `C` squared. We fix `N = 10` for the experiment, define the register offsets (input and the two resources), a random single-qubit input preparation `Cprep`, and helpers to apply it. The verification ancillas — one per stabilizer measurement — are added inside `build_cznr`.

In [ ]:
N = 10
R_in, R_res1, R_res2 = 0, N, 2 * N   # register offsets

def C_on(offset, n):
    """All-pairs controlled-Z on qubits offset..offset+n-1 (the graph-state circuit)."""
    c = stim.Circuit()
    for i in range(n):
        for j in range(i + 1, n):
            c.append('CZ', [offset + i, offset + j])
    return c

def make_Cprep(n, seed):
    """A random single-qubit Clifford layer, as per-qubit gate lists from {H, S, S_DAG}."""
    rng = np.random.default_rng(seed)
    return [[g for g in rng.choice(['H', 'S', 'S_DAG', 'I', 'I'], size=3) if g != 'I']
            for _ in range(n)]

def apply_Cprep(c, ops, offset, inverse=False):
    """Apply the Cprep layer (or its inverse) to qubits offset..offset+n-1."""
    for q, layer in enumerate(ops):
        for g in (layer[::-1] if inverse else layer):
            c.append({'H': 'H', 'S': 'S_DAG', 'S_DAG': 'S'}[g] if inverse else g, [offset + q])

# The single input preparation used throughout the experiment (N-qubit version).
Cprep = make_Cprep(N, seed=5)

# Bitstring convention for reading qBraid/IonQ counts back into qubit order. qubit 0 = LSB
# (calibrated on the simulator: this value decodes a noiseless run to exactly 0).
BIT_REVERSE = False

### Verification stabilizers

We use the same two families of graph-state stabilizers as the previous notebook: low-weight `Y_i Y_j`, and high-weight `X_i` times `Z_j` on every other qubit. A stabilizer chosen on the first resource is also measured on the second resource, shifted onto its qubits.

In [ ]:
def lowweight_stab(i, j, offset, n=N):
    """Low-weight stabilizer Y_i Y_j of the resource at the given offset (n qubits per register)."""
    p = stim.PauliString(3 * n); p[offset + i] = 2; p[offset + j] = 2
    return p

def highweight_stab(i, offset, n=N):
    """High-weight stabilizer X_i * product_{j != i} Z_j of the resource at the offset."""
    p = stim.PauliString(3 * n); p[offset + i] = 1
    for j in range(n):
        if j != i: p[offset + j] = 3
    return p

def shift_to_res2(stab, n=N):
    """Move a stabilizer defined on resource 1 onto resource 2 (shift indices by n)."""
    xs, zs = stab.to_numpy(); p = stim.PauliString(3 * n)
    for q in range(len(xs)):
        if xs[q] or zs[q]:
            p[q + n] = 2 if (xs[q] and zs[q]) else (1 if xs[q] else 3)
    return p

In [ ]:
def build_cznr(n, Cprep, stabs_res1, ancillas, draw=False):
    """Build the C-squared CZNR circuit as a Qiskit QuantumCircuit.

    Single source of truth for the circuit: drawn below at n = 3 for legibility and
    submitted to hardware at n = 10. No measurement gates are included, because IonQ
    through qBraid measures every qubit at the end automatically. With draw=True,
    stage barriers and a final measurement are added for the diagram only; they do
    not change the submitted circuit.

    Gate set: IonQ's supported gate set has cx but not cz, so for submission
    (draw=False) each cz is emitted as its h.cx.h decomposition and each cy as its
    sdg.cx.s decomposition. With draw=True the compact cz/cy gates are used instead,
    for a legible diagram. The two are logically identical (cz = H_b CX H_b), so the
    picture still faithfully shows the submitted circuit.

    n            qubits per register (input, resource 1, resource 2).
    Cprep        per-qubit input-preparation layer from make_Cprep(n, ...).
    stabs_res1   stabilizers checked on resource 1 (PauliStrings over 3n qubits);
                 the same ones, shifted, are checked on resource 2.
    ancillas     one qubit per check (2 * len(stabs_res1) total).

    Returns (circuit, measured_order); measured_order lists the qubit indices in the
    order IonQ returns them, so decode_and_correct can read off b0, b1, b2.
    """
    from qiskit import QuantumCircuit
    R_in, R_res1, R_res2 = 0, n, 2 * n
    n_stab = len(stabs_res1)
    total_qubits = 3 * n + 2 * n_stab
    qc = QuantumCircuit(total_qubits)

    def add_cz(a, b):
        """Compact cz for the diagram; h.cx.h decomposition for IonQ submission."""
        if draw:
            qc.cz(a, b)
        else:
            qc.h(b); qc.cx(a, b); qc.h(b)      # cz = H_b CX H_b

    def add_cy(a, b):
        """Compact cy for the diagram; sdg.cx.s decomposition for IonQ submission."""
        if draw:
            qc.cy(a, b)
        else:
            qc.sdg(b); qc.cx(a, b); qc.s(b)    # cy = S_b CX S_b^dagger

    def apply_prep(offset, inverse=False):
        """Apply the Cprep layer (or its inverse) with Qiskit gates."""
        for q, layer in enumerate(Cprep):
            for g in (layer[::-1] if inverse else layer):
                gate = {'H': 'H', 'S': 'S_DAG', 'S_DAG': 'S'}[g] if inverse else g
                if   gate == 'H':     qc.h(offset + q)
                elif gate == 'S':     qc.s(offset + q)
                elif gate == 'S_DAG': qc.sdg(offset + q)

    def verify(stab, a):
        """Read one stabilizer onto ancilla a, left idle (no measurement here)."""
        qc.h(a)
        xs, zs = stab.to_numpy()
        for q in range(3 * n):
            if   xs[q] and zs[q]: add_cy(a, q)
            elif xs[q]:           qc.cx(a, q)
            elif zs[q]:           add_cz(a, q)
        qc.h(a)

    # 1. Input preparation and the two resource graph states (H then all-pairs CZ).
    apply_prep(R_in)
    for i in range(n): qc.h(R_res1 + i)
    for i in range(n):
        for j in range(i + 1, n): add_cz(R_res1 + i, R_res1 + j)
    for i in range(n): qc.h(R_res2 + i)
    for i in range(n):
        for j in range(i + 1, n): add_cz(R_res2 + i, R_res2 + j)
    if draw: qc.barrier()

    # 2. Verify resource 1, then apply C by coupling resource 1 into the input.
    for k in range(n_stab): verify(stabs_res1[k], ancillas[k])
    if draw: qc.barrier()
    for i in range(n): qc.cx(R_res1 + i, R_in + i)
    if draw: qc.barrier()

    # 3. Verify resource 2 (shifted stabilizers), then apply C again
    #    (output lands on resource 2 = C squared applied to the input).
    for k in range(n_stab): verify(shift_to_res2(stabs_res1[k], n), ancillas[n_stab + k])
    if draw: qc.barrier()
    for i in range(n): qc.cx(R_res2 + i, R_res1 + i)
    if draw: qc.barrier()

    # 4. Undo the input preparation on resource 2, which now holds C^2 Cprep|0>.
    apply_prep(R_res2, inverse=True)

    # IonQ returns bitstrings with qubit 0 as the leftmost bit, in qubit-index order.
    measured_order = list(range(total_qubits))
    if draw:
        qc.barrier(); qc.measure_all()   # shown for orientation only; not submitted
    return qc, measured_order

### Drawn on three qubits

The diagram below shows `build_cznr` at three qubits per register with a single low-weight stabilizer per resource, so the five stages stay legible. The barriers separate the stages, and the final measurement layer is included for orientation only — it is omitted from the circuit actually submitted, since IonQ measures every qubit automatically.

In [ ]:
# Draw the exact circuit that gets submitted, at n = 3 with one stabilizer per resource.
n_draw = 3
Cprep_draw = make_Cprep(n_draw, seed=5)
stabs_draw = [lowweight_stab(0, 1, offset=n_draw, n=n_draw)]   # Y_0 Y_1 on resource 1
ancillas_draw = [3 * n_draw + k for k in range(2 * len(stabs_draw))]

qc_draw, _ = build_cznr(n_draw, Cprep_draw, stabs_draw, ancillas_draw, draw=True)
qc_draw.draw(output='mpl', fold=28, scale=0.6)

## The post-processing correction

After the circuit runs, each shot gives three n-bit strings, `b0` from the input register, `b1` from the first resource, and `b2` from the second resource, plus the verification bits. The verification bits decide whether the shot is kept: any non-zero verification bit means a fault was detected and the shot is discarded (a restart). The discarded fraction estimates the restart rate.

For a kept shot, the teleportation byproducts are corrected classically. The correction is written compactly as

$$P_{\text{corr}} = C_{\text{prep}} \, C \, X^{\vec{b}_1 \oplus \vec{b}_0} \, C \, C_{\text{prep}}^{\dagger} = X^{\vec{x}} Z^{\vec{z}},$$

and the corrected output is the input-register-free bitstring `b_CZNR = b2 XOR x`. A point specific to running `C` squared: the byproduct from the first application of `C` is created on resource 1 and then passes through the second application of `C`, while the byproduct from the second application is created directly on the output. The two therefore enter the correction at different points, and the implementation below reflects this by conjugating the `b0` part through one more copy of `C` than the `b1` part.

We evaluate the byproduct using the fact that for a CZ circuit, conjugating an `X` on qubit `i` by `C` gives `X_i` times `Z` on the graph neighbours of `i`. This is the same operator that appeared as the in-circuit correction in *CliNR in Practice: Reaching Breakeven with CZNR*.

In [ ]:
def make_correction(Cprep, n=N):
    """Return a function corr_x(b0, b1) -> x, the X-part of the post-processing correction,
    for the given input preparation. The output bitstring is then b2 XOR x.
    n is the number of qubits per register (defaults to the experiment's N)."""
    C_tableau = stim.Tableau.from_circuit(C_on(0, n))   # acts on one n-qubit block
    prep_circ = stim.Circuit(); apply_Cprep(prep_circ, Cprep, 0)
    P_tableau = stim.Tableau.from_circuit(prep_circ)

    def E1(i):
        """C X_i C^dagger for the all-pairs CZ circuit: X on i, Z on every other qubit."""
        p = stim.PauliString(n); p[i] = 1
        for j in range(n):
            if j != i: p[j] = 3
        return p

    def corr_x(b0, b1):
        total = stim.PauliString(n)
        # b0 byproduct passes through the second C: conjugate each term by C
        for i in range(n):
            if b0[i]: total = total * C_tableau(E1(i))
        # b1 byproduct is created directly on the output
        for i in range(n):
            if b1[i]: total = total * E1(i)
        # both are then referred back through the input preparation
        total = P_tableau.inverse()(total)
        xs, zs = total.to_numpy()
        return np.array([1 if xs[q] else 0 for q in range(n)], dtype=int)
    return corr_x

In [ ]:
def decode_and_correct(counts, measured_order, Cprep, n_verify, n=N, reverse_bits=None):
    """Apply the post-processing correction to returned counts.

    Each key is a binary string over the 3n+n_verify measured qubits. We parse it as a
    fixed-width integer -- qBraid strips leading zeros, so a key can be shorter than the
    full width, and integer parsing reads the missing high bits as 0 (no IndexError). We
    then read off b0 (input), b1 (resource 1), b2 (resource 2), and the verification bits.
    reverse_bits selects the qubit<->bit convention; None uses the global BIT_REVERSE.
    n is the number of qubits per register. Returns (logical_error, restart_rate)."""
    if reverse_bits is None:
        reverse_bits = BIT_REVERSE
    corr_x = make_correction(Cprep, n)
    total_qubits = 3 * n + n_verify

    def key_to_bits(key):
        val = int(key, 2)
        return [(val >> (total_qubits - 1 - q if reverse_bits else q)) & 1
                for q in range(total_qubits)]

    in_q  = list(range(0, n)); r1_q = list(range(n, 2 * n)); r2_q = list(range(2 * n, 3 * n))
    ver_q = list(range(3 * n, 3 * n + n_verify))
    n_fail = n_accept = n_zero = 0
    total = sum(counts.values())
    for key, count in counts.items():
        bits = key_to_bits(key)
        if any(bits[q] for q in ver_q):           # fault detected -> discard (restart)
            n_fail += count; continue
        b0 = np.array([bits[q] for q in in_q], dtype=int)
        b1 = np.array([bits[q] for q in r1_q], dtype=int)
        b2 = np.array([bits[q] for q in r2_q], dtype=int)
        x = corr_x(b0, b1)
        n_accept += count
        if not (b2 ^ x).any():
            n_zero += count
    logical_error = 1 - n_zero / n_accept if n_accept else 1.0
    return logical_error, n_fail / total

## Smoke test on the free simulator

Before spending anything on hardware, we validate the whole submission path — the `cx`-decomposed circuit *and* the post-processing decoder — on the free `ionq:ionq:sim:simulator`, at a small `n = 2`. We run both stabilizer families: high-weight (whose `Z` couplings use the `cz -> h.cx.h` decomposition) and low-weight `Y_iY_j` (which exercises the `cy -> sdg.cx.s` decomposition). Noiseless, the corrected logical error must be exactly zero — that confirms the gate decompositions, the readback, and the decoder all agree with the circuit. Each family is then run once more under IonQ's `aria-1` noise model.

Counts are read back with fixed-width integer parsing: qBraid strips leading zeros, so a returned key can be shorter than `3n + checks`, and integer parsing reads the missing high bits as zero (using the `BIT_REVERSE` convention). Explicit measurements would be silently ignored by the IonQ conversion, so we don't add them.

In [ ]:
from qbraid import QbraidProvider

def sim_check(n, family, shots=800, noise_model=None):
    """Run the cx-decomposed CZNR circuit + decoder on the free ionq:ionq simulator.
    family 'high-weight' exercises the cz->cx path; 'low-weight' the cy path.
    Returns (logical_error, restart)."""
    Cp = make_Cprep(n, seed=5)
    if family == 'high-weight':
        stabs = [highweight_stab(0, n, n=n)]            # X_0 * Z... on resource 1
    else:
        stabs = [lowweight_stab(0, 1, offset=n, n=n)]   # Y_0 Y_1 on resource 1
    anc = [3 * n + k for k in range(2 * len(stabs))]
    n_verify = 2 * len(stabs)
    qc, order = build_cznr(n, Cp, stabs, anc)           # draw=False -> cx / cy decomposed

    device = QbraidProvider().get_device('ionq:ionq:sim:simulator')
    run_kwargs = {'shots': shots}
    if noise_model is not None:
        run_kwargs['runtime_options'] = {'noise': {'model': noise_model, 'seed': 42}}
    job = device.run(qc, **run_kwargs)
    job.wait_for_final_state()
    counts = job.result().data.measurement_counts
    return decode_and_correct(counts, order, Cp, n_verify, n=n)

print(f'{"family":12} {"noise":8} {"logical error":>13} {"restart":>8}')
print('-' * 45)
for family in ['high-weight', 'low-weight']:
    err0, r0 = sim_check(2, family, noise_model=None)
    print(f'{family:12} {"none":8} {err0:>13.4f} {r0:>8.3f}')
    assert err0 == 0.0, f'{family}: noiseless logical error must be 0, got {err0}'
    try:
        errn, rn = sim_check(2, family, noise_model='aria-1')
        print(f'{family:12} {"aria-1":8} {errn:>13.4f} {rn:>8.3f}')
    except Exception as e:
        print(f'{family:12} {"aria-1":8}  (skipped: {type(e).__name__})')
print('\nNoiseless decode is exact (0.0) for both families: gate decomposition + readback are correct.')

## Running on IonQ Forte through qBraid

The submission is controlled by two flags. `SUBMIT` gates whether any job is sent at all (default `False`, so the notebook is safe to run end to end). `RUN_ON_HARDWARE` chooses the target: `False` sends to the free `ionq:ionq:sim:simulator`, `True` sends to paid IonQ Forte (`aws:ionq:qpu:forte-1`). Set `RUN_ON_HARDWARE = True` only with a qBraid account that has IonQ access and credits.

Two facts shape the pipeline. First, IonQ through qBraid measures every qubit at the end automatically and does not accept measurement statements, which matches our circuit since `build_cznr` (draw=False) already omits them. Second, the AWS Forte path accepts a strict gate set — `cx` but not `cz` — so `build_cznr` decomposes every `cz` into `h.cx.h` (and `cy` into `sdg.cx.s`) for submission. The free simulator accepts a wider set, but we submit the same cx-decomposed circuit either way.

Submission is split into two cells so an interrupted wait never orphans a paid job: the first builds and submits, printing and saving the job id immediately; the second blocks on the result and decodes it with `decode_and_correct`. On the simulator we apply IonQ's `aria-1` noise model for a realistic run; hardware carries its own physical noise. (A full n=10 / 36-qubit noisy simulation is heavy — revisit before relying on it.)

### A note on credits

A run on IonQ Forte via qBraid uses qBraid credits (1 credit = $0.01). The AWS pricing model charges **30 credits per task + 8 credits per shot** — cost scales with shot count, not gate count. At 2048 shots that is 16,414 credits (~$164). The free `ionq:ionq:sim:simulator` costs nothing; always validate there (and on the smoke test above) before submitting to hardware. Confirm current rates at [docs.qbraid.com/v2/home/pricing](https://docs.qbraid.com/v2/home/pricing).

In [ ]:
# --- Submission controls ---
# SUBMIT:          master gate. Leave False to skip all job submission (safe default).
# RUN_ON_HARDWARE: False -> free ionq:ionq simulator; True -> paid IonQ Forte (aws).
SUBMIT = False
RUN_ON_HARDWARE = False

# The free simulator accepts an expanded gate set; the AWS Forte hardware is strict
# (cx, not cz), which is why build_cznr decomposes cz->cx / cy for submission (draw=False).
IONQ_DEVICE = 'aws:ionq:qpu:forte-1' if RUN_ON_HARDWARE else 'ionq:ionq:sim:simulator'
HARDWARE_SHOTS = 100

# Noise: apply IonQ's aria-1 model on the simulator for a realistic run; hardware has its
# own physical noise, so leave it off there.
# NOTE: a full n=10 / 36-qubit aria-1 simulation is heavy and slow -- revisit before relying
# on it (a smaller n, or the smoke test above, is the practical dry-run).
IONQ_NOISE = None if RUN_ON_HARDWARE else 'aria-1' 

In [ ]:
def submit_to_ionq(Cprep, stabs_res1, ancillas, device_id, shots, noise_model=None):
    """Build the CZNR submission circuit and submit it through qBraid's runtime.

    Returns (job, measured_order) WITHOUT blocking, so the job id is captured before any
    wait -- an interrupted wait then never orphans the job. Retrieve results separately with
    job.wait_for_final_state() and job.result().

    device_id    'ionq:ionq:sim:simulator' (free) or 'aws:ionq:qpu:forte-1' (paid Forte).
    noise_model  IonQ noise model name (e.g. 'aria-1') to apply on a simulator; leave None
                 on real hardware, which carries its own physical noise."""
    from qbraid import QbraidProvider

    # Same builder drawn above, at n = N. draw=False emits cx (IonQ's set has no cz).
    qc, measured_order = build_cznr(N, Cprep, stabs_res1, ancillas)

    provider = QbraidProvider()
    device = provider.get_device(device_id)
    run_kwargs = {'shots': shots}
    if noise_model is not None:
        run_kwargs['runtime_options'] = {'noise': {'model': noise_model, 'seed': 42}}
    job = device.run(qc, **run_kwargs)
    print(f'Job submitted to {device_id}. Job ID: {job.id}')
    return job, measured_order

In [ ]:
# Build the n = 10 submission circuit (high-weight, r = 3) and submit it -- no blocking.
hw_centres = [0, 3, 7]
hw_stabs = [highweight_stab(i, R_res1) for i in hw_centres]
hw_ancillas = [3 * N + k for k in range(2 * len(hw_stabs))]
hw_nverify = 2 * len(hw_stabs)

if SUBMIT:
    hw_job, measured_order = submit_to_ionq(
        Cprep, hw_stabs, hw_ancillas, IONQ_DEVICE, HARDWARE_SHOTS, noise_model=IONQ_NOISE
    )
    JOB_ID = hw_job.id
    print(f'Saved JOB_ID = {JOB_ID}')
    print('Retrieve results in the next cell; the job id above survives an interrupted wait.')
else:
    hw_job, measured_order, JOB_ID = None, None, None
    print(f'Submission off (SUBMIT = False). Set SUBMIT = True to submit to {IONQ_DEVICE}.')

In [ ]:
# Retrieve and decode. Run this after the job above completes.
# Interrupting the wait here is safe: the job keeps running and JOB_ID (printed above) still
# identifies it, so you can re-run this cell -- or, in a fresh kernel, reload the job from
# JOB_ID via the Jobs panel / QbraidProvider before decoding.
if hw_job is not None:
    hw_job.wait_for_final_state()
    counts = hw_job.result().data.measurement_counts
    hw_error, hw_restart = decode_and_correct(counts, measured_order, Cprep, hw_nverify)
    print(f'CZNR logical error on {IONQ_DEVICE}: {hw_error:.5f}  (restart {hw_restart:.3f})')
else:
    counts = None
    print('No job to retrieve. Set SUBMIT = True, run the submission cell, then this one.')

### Retrieve a job by ID (fresh session)

If you submitted in one session and the kernel is gone, come back later, run the **definition cells above** (Setup, Building blocks, stabilizer helpers, `build_cznr`, `make_correction`, `decode_and_correct`), then paste the job id from the qBraid **Jobs panel** into the cell below. It reconstructs the job from its id alone — nothing from the submission session is needed — waits for it to finish, decodes it, and sets `counts` so the progressive-ignoring analysis below runs.

Make sure `N` and `hw_centres` here match the run you submitted (the decoder is `N`-dependent). The reconstruction tries the known qBraid entry points; if none match your installed version, it prints the available job methods so you can tell which one to use.

In [ ]:
# --- Standalone retrieval by job id (works in a fresh kernel / new session) ---
# Paste the job id copied from the qBraid Jobs panel. Requires only the definition cells
# above to have been run (they rebuild Cprep, the stabilizers, and the decoder). N and
# hw_centres must match the submitted run.
JOB_ID = "PASTE_JOB_ID_HERE"

def retrieve_job(job_id):
    """Reconstruct a qBraid job from its id string, across qbraid-runtime versions.
    Tries the known provider entry points, then the native job class; if none match this
    version, raises with the list of job-related methods actually available."""
    from qbraid.runtime import QbraidProvider
    provider = QbraidProvider()
    tried = []
    for name in ('get_job', 'load_job', 'retrieve_job'):          # provider entry points
        fn = getattr(provider, name, None)
        if fn is None:
            continue
        try:
            job = fn(job_id)
            if hasattr(job, 'wait_for_final_state'):
                return job
        except Exception as e:
            tried.append(f'{name}(): {type(e).__name__}: {e}')
    try:                                                          # fall back to the job class
        from qbraid.runtime.native import QbraidJob
        try:
            return QbraidJob(job_id)
        except TypeError:
            return QbraidJob(job_id, provider.session)
    except Exception as e:
        tried.append(f'QbraidJob(): {type(e).__name__}: {e}')
    methods = [m for m in dir(provider) if 'job' in m.lower()]
    raise RuntimeError('Could not reconstruct the job. Tried:\n  ' + '\n  '.join(tried) +
                       f'\nProvider job-related methods in this version: {methods}')

if JOB_ID.strip() in ('', 'PASTE_JOB_ID_HERE'):
    print('Paste a job id into JOB_ID above, then re-run this cell.')
    if 'counts' not in globals():
        counts = None
else:
    # Rebuild the run parameters (deterministic; must match the submitted circuit).
    hw_centres = [0, 3, 7]
    hw_stabs = [highweight_stab(i, R_res1) for i in hw_centres]
    hw_nverify = 2 * len(hw_stabs)
    measured_order = list(range(3 * N + hw_nverify))

    job = retrieve_job(JOB_ID.strip())
    print(f'Retrieved {getattr(job, "id", JOB_ID)} | status: {job.status()}')
    job.wait_for_final_state()
    counts = job.result().data.measurement_counts
    hw_error, hw_restart = decode_and_correct(counts, measured_order, Cprep, hw_nverify)
    print(f'CZNR logical error: {hw_error:.5f}  (restart {hw_restart:.3f})')

## Progressive stabilizer ignoring

This analysis reproduces the paper's demonstration that the verification is doing real work, and it runs on hardware results. Starting from a run with three stabilizers, we recompute the logical error rate while ignoring some of the stabilizer outcomes, that is, accepting shots that those checks would have rejected. As more stabilizers are ignored, fewer shots are discarded, so the acceptance rate rises, and the logical error rate gets worse because faults that would have been caught are now kept. A logical error rate that climbs as checks are dropped is direct evidence that the checks were removing real errors.

This requires a hardware run with three stabilizers, so it activates only when a run is available from the section above. With hardware switched off, it reports that there is nothing to analyse.

In [ ]:
def progressive_ignore(counts, measured_order, Cprep, n_total_checks, n=N, reverse_bits=None):
    # Dropping stabilizer checks means accepting shots that would have been rejected.
    # If the logical error rate rises as checks are dropped, the checks were catching
    # real errors -- the signature that verification is doing useful work.
    """Recompute logical error and acceptance while ignoring the last k of n_total_checks
    stabilizer measurements, for k = 0..n_total. Returns (k_ignored, logical_error, acceptance)."""
    if reverse_bits is None:
        reverse_bits = BIT_REVERSE
    corr_x = make_correction(Cprep, n)
    total_qubits = 3 * n + n_total_checks

    def key_to_bits(key):
        val = int(key, 2)
        return [(val >> (total_qubits - 1 - q if reverse_bits else q)) & 1
                for q in range(total_qubits)]

    in_q = list(range(0, n)); r1_q = list(range(n, 2 * n)); r2_q = list(range(2 * n, 3 * n))
    ver_q = list(range(3 * n, 3 * n + n_total_checks))
    total = sum(counts.values())
    out = []
    for k in range(n_total_checks + 1):
        active = ver_q[:n_total_checks - k]          # checks still enforced
        n_accept = n_zero = 0
        for key, count in counts.items():
            bits = key_to_bits(key)
            if any(bits[q] for q in active):
                continue
            b0 = np.array([bits[q] for q in in_q], dtype=int)
            b1 = np.array([bits[q] for q in r1_q], dtype=int)
            b2 = np.array([bits[q] for q in r2_q], dtype=int)
            x = corr_x(b0, b1); n_accept += count
            if not (b2 ^ x).any(): n_zero += count
        logical_error = 1 - n_zero / n_accept if n_accept else 1.0
        out.append((k, logical_error, n_accept / total))
    return out

if counts is not None:
    rows = progressive_ignore(counts, measured_order, Cprep, hw_nverify)
    print(f'{"ignored":>8} | {"logical error":>13} | {"acceptance":>10}')
    print('-' * 38)
    for k, err, acc in rows:
        print(f'{k:>8} | {err:>13.5f} | {acc:>10.3f}')
    ks = [r[0] for r in rows]; errs = [r[1] for r in rows]; accs = [r[2] for r in rows]
    fig, ax1 = plt.subplots(figsize=(6.5, 4))
    ax1.plot(ks, errs, 'o-', color='tab:red'); ax1.set_xlabel('stabilizers ignored')
    ax1.set_ylabel('logical error rate', color='tab:red')
    ax2 = ax1.twinx(); ax2.plot(ks, accs, 's--', color='tab:blue')
    ax2.set_ylabel('acceptance rate', color='tab:blue')
    ax1.set_title('Effect of ignoring stabilizers')
    plt.tight_layout(); plt.show()
else:
    print('No results to analyse. Run the submission section above first.')

## What we have

CZNR by post-processing implements the same protocol as the in-circuit version of *CliNR in Practice: Reaching Breakeven with CZNR*, but defers every correction to classical post-processing. That is what lets it run on hardware with no mid-circuit measurement; the previous notebook's simulations show the deferral costs nothing in logical error rate.

The hardware section submits the same circuit to IonQ Forte through qBraid when enabled, and the progressive stabilizer-ignoring analysis, run on the hardware results, shows the verification removing real errors. Together these reproduce the breakeven demonstration of the experiment.

This is where the present series ends. A natural next step is the noise reduction scheme based on mid-circuit measurements of Brown and collaborators, which uses the measurement capabilities we set aside here.

## References

[1] N. Delfosse and E. Tham, "Clifford Noise Reduction," arXiv:2407.06583 (2024).

[2] E. Tham and N. Delfosse, "Optimized Clifford Noise Reduction: Theory, Simulations and Experiments," *Quantum* **9**, 1829 (2025). [arXiv:2504.13356](https://arxiv.org/abs/2504.13356)